In [1]:
import pandas as pd
import numpy as np
import xlsxwriter as xw

In [2]:
file = "Dados_PowerBI.xlsx"
xls = pd.ExcelFile(file)

In [3]:
xls.sheet_names

['Exercício',
 'Sudeste',
 'Norte e Nordeste',
 'Centro-Oeste e Sul',
 'Tabela Final']

In [4]:
sudeste = pd.read_excel(file, sheet_name="Sudeste")

In [5]:
sudeste.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,SUDESTE,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
0,NaN,NaN,NaN,NaN,NaN,NaN,SP Capital,NaN,NaN,SP Interior 1,NaN,NaN,SP Interior 2,NaN,NaN,Sudeste capital,NaN,NaN,Sudeste interior,NaN
1,KM,Risco,Utilitario,Van,Vuc,0-60,61-90,>91,0-60,61-90,>91,0-60,61-90,>91,0-60,61-90,>91,0-60,61-90,>91
2,1-100,255,320,352,405,0.35,2.3,1.05,0.5,2,0.8,0.5,2.7,1.4,0.6,1.9,0.7,0.5,1.5,0.7
3,101-150,295,368,405,455,0.35,2.3,1.05,0.5,2,0.8,0.5,2.7,1.4,0.6,1.9,0.7,0.5,1.5,0.7
4,151-200,335,416,458,505,0.35,2.3,1.05,0.5,2,0.8,0.5,2.7,1.4,0.6,1.9,0.7,0.5,1.5,0.7


In [6]:
zonas = sudeste.iloc[0]

cabecalho = sudeste.iloc[1]

dados = sudeste.iloc[2:].reset_index(drop=True)

dados.columns = cabecalho

dados.head()

1,KM,Risco,Utilitario,Van,Vuc,0-60,61-90,>91,0-60,61-90,>91,0-60,61-90,>91,0-60,61-90,>91,0-60,61-90,>91
0,1-100,255,320,352,405,0.35,2.3,1.05,0.5,2,0.8,0.5,2.7,1.4,0.6,1.9,0.7,0.5,1.5,0.7
1,101-150,295,368,405,455,0.35,2.3,1.05,0.5,2,0.8,0.5,2.7,1.4,0.6,1.9,0.7,0.5,1.5,0.7
2,151-200,335,416,458,505,0.35,2.3,1.05,0.5,2,0.8,0.5,2.7,1.4,0.6,1.9,0.7,0.5,1.5,0.7
3,201-300,375,464,511,555,0.35,2.3,1.05,0.5,2,0.8,0.5,2.7,1.4,0.6,1.9,0.7,0.5,1.5,0.7
4,>301,415,512,564,605,0.35,2.3,1.05,0.5,2,0.8,0.5,2.7,1.4,0.6,1.9,0.7,0.5,1.5,0.7


In [7]:
sp_capital = dados[['KM', 'Risco', 'Utilitario', 'Van', 'Vuc']].copy()

sp_capital = sp_capital.dropna(subset=['KM'])

sp_capital = sp_capital[sp_capital['KM'].astype(str).str.contains('-|>')]

print(sp_capital.head())

sp_capital_long = sp_capital.melt(
    id_vars='KM',
    var_name='Veículo',
    value_name='Custo Fixo'
)

print(sp_capital_long.head(10))

1       KM Risco Utilitario  Van  Vuc
0    1-100   255        320  352  405
1  101-150   295        368  405  455
2  151-200   335        416  458  505
3  201-300   375        464  511  555
4     >301   415        512  564  605
        KM     Veículo Custo Fixo
0    1-100       Risco        255
1  101-150       Risco        295
2  151-200       Risco        335
3  201-300       Risco        375
4     >301       Risco        415
5    1-100  Utilitario        320
6  101-150  Utilitario        368
7  151-200  Utilitario        416
8  201-300  Utilitario        464
9     >301  Utilitario        512


In [8]:
sp_capital_long['Região'] = 'Sudeste'
sp_capital_long['Zona'] = 'SP Capital'

sp_capital_long['KM Inicial'] = sp_capital_long['KM'].apply(
    lambda x: int(str(x).replace('>', '').split('-')[0])
)

sp_capital_long['KM Final'] = sp_capital_long['KM'].apply(
    lambda x: 99999 if '>' in str(x) else int(str(x).split('-')[1])
)

sp_capital_long = sp_capital_long[
    ['Região', 'Zona', 'KM Inicial', 'KM Final', 'Veículo', 'Custo Fixo']
]

print(sp_capital_long.head(10))

    Região        Zona  KM Inicial  KM Final     Veículo Custo Fixo
0  Sudeste  SP Capital           1       100       Risco        255
1  Sudeste  SP Capital         101       150       Risco        295
2  Sudeste  SP Capital         151       200       Risco        335
3  Sudeste  SP Capital         201       300       Risco        375
4  Sudeste  SP Capital         301     99999       Risco        415
5  Sudeste  SP Capital           1       100  Utilitario        320
6  Sudeste  SP Capital         101       150  Utilitario        368
7  Sudeste  SP Capital         151       200  Utilitario        416
8  Sudeste  SP Capital         201       300  Utilitario        464
9  Sudeste  SP Capital         301     99999  Utilitario        512


In [9]:
list(dados.columns)

['KM',
 'Risco',
 'Utilitario',
 'Van',
 'Vuc',
 '0-60',
 '61-90',
 '>91',
 '0-60',
 '61-90',
 '>91',
 '0-60',
 '61-90',
 '>91',
 '0-60',
 '61-90',
 '>91',
 '0-60',
 '61-90',
 '>91']

In [10]:
dados.columns = [
    'KM',
    'Risco', 'Utilitario', 'Van', 'Vuc',
    'SPCap_0_60', 'SPCap_61_90', 'SPCap_91',
    'SPInt1_0_60', 'SPInt1_61_90', 'SPInt1_91',
    'SPInt2_0_60', 'SPInt2_61_90', 'SPInt2_91',
    'SudCap_0_60', 'SudCap_61_90', 'SudCap_91',
    'SudInt_0_60', 'SudInt_61_90', 'SudInt_91'
]

print(dados.head())

        KM Risco Utilitario  Van  Vuc SPCap_0_60 SPCap_61_90 SPCap_91  \
0    1-100   255        320  352  405       0.35         2.3     1.05   
1  101-150   295        368  405  455       0.35         2.3     1.05   
2  151-200   335        416  458  505       0.35         2.3     1.05   
3  201-300   375        464  511  555       0.35         2.3     1.05   
4     >301   415        512  564  605       0.35         2.3     1.05   

  SPInt1_0_60 SPInt1_61_90 SPInt1_91 SPInt2_0_60 SPInt2_61_90 SPInt2_91  \
0         0.5            2       0.8         0.5          2.7       1.4   
1         0.5            2       0.8         0.5          2.7       1.4   
2         0.5            2       0.8         0.5          2.7       1.4   
3         0.5            2       0.8         0.5          2.7       1.4   
4         0.5            2       0.8         0.5          2.7       1.4   

  SudCap_0_60 SudCap_61_90 SudCap_91 SudInt_0_60 SudInt_61_90 SudInt_91  
0         0.6          1.9       0.7

In [11]:
zonas_sudeste = [
    'SP Capital',
    'SP Interior 1',
    'SP Interior 2',
    'Sudeste capital',
    'Sudeste interior'
]

lista_sudeste = []

for zona in zonas_sudeste:
    temp = sp_capital_long.copy()
    temp['Zona'] = zona
    lista_sudeste.append(temp)

sudeste_final = pd.concat(lista_sudeste, ignore_index=True)

print(sudeste_final.head(50))

print(sudeste_final['Zona'].unique())

     Região           Zona  KM Inicial  KM Final     Veículo Custo Fixo
0   Sudeste     SP Capital           1       100       Risco        255
1   Sudeste     SP Capital         101       150       Risco        295
2   Sudeste     SP Capital         151       200       Risco        335
3   Sudeste     SP Capital         201       300       Risco        375
4   Sudeste     SP Capital         301     99999       Risco        415
5   Sudeste     SP Capital           1       100  Utilitario        320
6   Sudeste     SP Capital         101       150  Utilitario        368
7   Sudeste     SP Capital         151       200  Utilitario        416
8   Sudeste     SP Capital         201       300  Utilitario        464
9   Sudeste     SP Capital         301     99999  Utilitario        512
10  Sudeste     SP Capital           1       100         Van        352
11  Sudeste     SP Capital         101       150         Van        405
12  Sudeste     SP Capital         151       200         Van    

In [12]:
norte_nordeste = pd.read_excel(file, sheet_name='Norte e Nordeste')

cabecalho_nn = norte_nordeste.iloc[5]

dados_nn = norte_nordeste.iloc[6:].reset_index(drop=True)
dados_nn.columns = cabecalho_nn

nn_capital = dados_nn[['KM', 'Risco', 'Utilitario', 'Van', 'Vuc']].copy()

nn_capital = nn_capital.dropna(subset=['KM'])
nn_capital = nn_capital[nn_capital['KM'].astype(str).str.contains('-|>')]

nn_capital_long = nn_capital.melt(
    id_vars='KM',
    var_name='Veículo',
    value_name='Custo Fixo'
)

nn_capital_long['Região'] = 'Norte/Nordeste'
nn_capital_long['Zona'] = 'Capital'

nn_capital_long['KM Inicial'] = nn_capital_long['KM'].apply(
    lambda x: int(str(x).replace('>', '').split('-')[0])
)

nn_capital_long['KM Final'] = nn_capital_long['KM'].apply(
    lambda x: 99999 if '>' in str(x) else int(str(x).split('-')[1])
)

nn_capital_long = nn_capital_long[
    ['Região', 'Zona', 'KM Inicial', 'KM Final', 'Veículo', 'Custo Fixo']
]

print(nn_capital_long.head(50))

            Região     Zona  KM Inicial  KM Final     Veículo Custo Fixo
0   Norte/Nordeste  Capital           1       100       Risco        255
1   Norte/Nordeste  Capital         101       150       Risco        295
2   Norte/Nordeste  Capital         151       200       Risco        335
3   Norte/Nordeste  Capital         201       300       Risco        375
4   Norte/Nordeste  Capital         301     99999       Risco        415
5   Norte/Nordeste  Capital           1       100  Utilitario        320
6   Norte/Nordeste  Capital         101       150  Utilitario        368
7   Norte/Nordeste  Capital         151       200  Utilitario        416
8   Norte/Nordeste  Capital         201       300  Utilitario        464
9   Norte/Nordeste  Capital         301     99999  Utilitario        512
10  Norte/Nordeste  Capital           1       100         Van        352
11  Norte/Nordeste  Capital         101       150         Van        405
12  Norte/Nordeste  Capital         151       200  

In [13]:
# Norte Capital
norte_capital = nn_capital_long.copy()
norte_capital['Região'] = 'Norte'
norte_capital['Zona'] = 'Capital'

# Nordeste Capital
nordeste_capital = nn_capital_long.copy()
nordeste_capital['Região'] = 'Nordeste'
nordeste_capital['Zona'] = 'Capital'

# Nordeste Interior
nordeste_interior = nn_capital_long.copy()
nordeste_interior['Região'] = 'Nordeste'
nordeste_interior['Zona'] = 'Interior'

norte_nordeste_final = pd.concat([
    norte_capital,
    nordeste_capital,
    nordeste_interior
], ignore_index=True)

print(norte_nordeste_final.head())
print(norte_nordeste_final[['Região','Zona']].drop_duplicates())

  Região     Zona  KM Inicial  KM Final Veículo Custo Fixo
0  Norte  Capital           1       100   Risco        255
1  Norte  Capital         101       150   Risco        295
2  Norte  Capital         151       200   Risco        335
3  Norte  Capital         201       300   Risco        375
4  Norte  Capital         301     99999   Risco        415
      Região      Zona
0      Norte   Capital
20  Nordeste   Capital
40  Nordeste  Interior


In [14]:
centro_sul = pd.read_excel(file, sheet_name='Centro-Oeste e Sul')
print(centro_sul.head(20))

    Unnamed: 0  Unnamed: 1 Unnamed: 2 Unnamed: 3  Unnamed: 4 Unnamed: 5  \
0          NaN         NaN        NaN        NaN         NaN        NaN   
1          NaN         NaN        NaN        NaN         NaN        NaN   
2          NaN         NaN        NaN        NaN         NaN        NaN   
3          NaN         NaN        NaN        NaN         NaN        NaN   
4          NaN         NaN        NaN        NaN         NaN        NaN   
5          NaN         NaN         KM      Risco  Utilitario        Van   
6          NaN         NaN      1-100        255         320        352   
7          NaN         NaN    101-150        295         368        405   
8          NaN         NaN    151-200        335         416        458   
9          NaN         NaN    201-300        375         464        511   
10         NaN         NaN       >301        415         512        564   

   Unnamed: 6 Unnamed: 7 Unnamed: 8 Unnamed: 9   Unnamed: 10 Unnamed: 11  \
0         NaN        Na

In [15]:
centro_sul = pd.read_excel(file, sheet_name='Centro-Oeste e Sul')

cabecalho_cs = centro_sul.iloc[5]

dados_cs = centro_sul.iloc[6:].reset_index(drop=True)
dados_cs.columns = cabecalho_cs

cs_capital = dados_cs[['KM', 'Risco', 'Utilitario', 'Van', 'Vuc']].copy()

cs_capital = cs_capital.dropna(subset=['KM'])
cs_capital = cs_capital[cs_capital['KM'].astype(str).str.contains('-|>')]

cs_capital_long = cs_capital.melt(
    id_vars='KM',
    var_name='Veículo',
    value_name='Custo Fixo'
)

cs_capital_long['Região'] = 'Centro-Oeste/Sul'
cs_capital_long['Zona'] = 'Capital'

cs_capital_long['KM Inicial'] = cs_capital_long['KM'].apply(
    lambda x: int(str(x).replace('>', '').split('-')[0])
)

cs_capital_long['KM Final'] = cs_capital_long['KM'].apply(
    lambda x: 99999 if '>' in str(x) else int(str(x).split('-')[1])
)

cs_capital_long = cs_capital_long[
    ['Região', 'Zona', 'KM Inicial', 'KM Final', 'Veículo', 'Custo Fixo']
]

print(cs_capital_long.head())

             Região     Zona  KM Inicial  KM Final Veículo Custo Fixo
0  Centro-Oeste/Sul  Capital           1       100   Risco        255
1  Centro-Oeste/Sul  Capital         101       150   Risco        295
2  Centro-Oeste/Sul  Capital         151       200   Risco        335
3  Centro-Oeste/Sul  Capital         201       300   Risco        375
4  Centro-Oeste/Sul  Capital         301     99999   Risco        415


In [16]:
# Centro-Oeste Capital
co_capital = cs_capital_long.copy()
co_capital['Região'] = 'Centro-Oeste'
co_capital['Zona'] = 'Capital'

# Centro-Oeste Interior
co_interior = cs_capital_long.copy()
co_interior['Região'] = 'Centro-Oeste'
co_interior['Zona'] = 'Interior'

# Sul Capital
sul_capital = cs_capital_long.copy()
sul_capital['Região'] = 'Sul'
sul_capital['Zona'] = 'Capital'

# Sul Interior
sul_interior = cs_capital_long.copy()
sul_interior['Região'] = 'Sul'
sul_interior['Zona'] = 'Interior'

centro_sul_final = pd.concat([
    co_capital,
    co_interior,
    sul_capital,
    sul_interior
], ignore_index=True)

print(centro_sul_final.head(50))
print(centro_sul_final[['Região','Zona']].drop_duplicates())

          Região      Zona  KM Inicial  KM Final     Veículo Custo Fixo
0   Centro-Oeste   Capital           1       100       Risco        255
1   Centro-Oeste   Capital         101       150       Risco        295
2   Centro-Oeste   Capital         151       200       Risco        335
3   Centro-Oeste   Capital         201       300       Risco        375
4   Centro-Oeste   Capital         301     99999       Risco        415
5   Centro-Oeste   Capital           1       100  Utilitario        320
6   Centro-Oeste   Capital         101       150  Utilitario        368
7   Centro-Oeste   Capital         151       200  Utilitario        416
8   Centro-Oeste   Capital         201       300  Utilitario        464
9   Centro-Oeste   Capital         301     99999  Utilitario        512
10  Centro-Oeste   Capital           1       100         Van        352
11  Centro-Oeste   Capital         101       150         Van        405
12  Centro-Oeste   Capital         151       200         Van    

In [ ]:
tabela_final = pd.concat([
    sudeste_final,
    norte_nordeste_final,
    centro_sul_final
], ignore_index=True)

print(tabela_final.head(500))
print(len(pd))

      Região        Zona  KM Inicial  KM Final Veículo Custo Fixo
0    Sudeste  SP Capital           1       100   Risco        255
1    Sudeste  SP Capital         101       150   Risco        295
2    Sudeste  SP Capital         151       200   Risco        335
3    Sudeste  SP Capital         201       300   Risco        375
4    Sudeste  SP Capital         301     99999   Risco        415
..       ...         ...         ...       ...     ...        ...
235      Sul    Interior           1       100     Vuc        405
236      Sul    Interior         101       150     Vuc        455
237      Sul    Interior         151       200     Vuc        505
238      Sul    Interior         201       300     Vuc        555
239      Sul    Interior         301     99999     Vuc        605

[240 rows x 6 columns]
240


In [18]:
tarifas = pd.DataFrame({
    'Região': [
        'SP Capital',
        'SP Interior 1',
        'SP Interior 2',
        'Sudeste capital',
        'Sudeste interior',
        'Norte Capital',
        'Nordeste Capital',
        'Nordeste Interior',
        'Centro-Oeste Capital',
        'Centro-Oeste Interior',
        'Sul Capital',
        'Sul Interior'
    ],
    'Valor_0_60': [
        0.35, 0.50, 0.50, 0.60, 0.50,
        0.55, 0.40, 0.25,
        0.65, 0.50,
        0.65, 0.65
    ],
    'Valor_61_90': [
        2.30, 2.00, 2.70, 1.90, 1.50,
        1.95, 1.00, 0.80,
        2.20, 1.30,
        2.00, 1.50
    ],
    'Valor_91': [
        1.05, 0.80, 1.40, 0.70, 0.70,
        0.70, 0.50, 0.50,
        0.90, 0.60,
        0.75, 0.75
    ]
})

In [19]:
casos = pd.DataFrame({
    'KM': [248, 111, 399, 300, 150, 123, 2, 245, 56, 84],
    'Região': [
        'Sudeste Capital',
        'Sul Interior',
        'SP Interior 1',
        'SP Capital',
        'Centro-Oeste Capital',
        'Centro-Oeste Interior',
        'Sul Capital',
        'Sul Interior',
        'Sudeste Interior',
        'SP Interior 2'
    ],
    'Veículo': [
        'Van',
        'Risco',
        'Vuc',
        'Utilitario',
        'Vuc',
        'Utilitario',
        'Risco',
        'Utilitario',
        'Van',
        'Utilitario'
    ],
    'Paradas': [36, 62, 111, 74, 10, 150, 122, 60, 61, 59]
})

In [20]:
casos['Veículo'] = casos['Veículo'].replace({
    'Utilitário': 'Utilitario'
})

tabela_final['Veículo'] = tabela_final['Veículo'].replace({
    'Utilitário': 'Utilitario'
})

def criar_chave(row):
    zona = str(row['Zona']).strip()
    regiao = str(row['Região']).strip()
    
    if zona.startswith('SP'):
        return zona
    
    if zona.lower().startswith(regiao.lower()):
        return zona
    
    return f"{regiao} {zona}"

tabela_final['Chave'] = tabela_final.apply(criar_chave, axis=1)

casos['Chave'] = casos['Região'].str.strip()

casos['Chave'] = casos['Chave'].replace({
    'Sudeste Capital': 'Sudeste capital',
    'Sudeste Interior': 'Sudeste interior'
})

print("Chaves da tabela_final:")
print(sorted(tabela_final['Chave'].unique()))

print("\nChaves dos casos:")
print(sorted(casos['Chave'].unique()))

Chaves da tabela_final:
['Centro-Oeste Capital', 'Centro-Oeste Interior', 'Nordeste Capital', 'Nordeste Interior', 'Norte Capital', 'SP Capital', 'SP Interior 1', 'SP Interior 2', 'Sudeste capital', 'Sudeste interior', 'Sul Capital', 'Sul Interior']

Chaves dos casos:
['Centro-Oeste Capital', 'Centro-Oeste Interior', 'SP Capital', 'SP Interior 1', 'SP Interior 2', 'Sudeste capital', 'Sudeste interior', 'Sul Capital', 'Sul Interior']


In [21]:
def calcular_linha(linha):
    km = linha['KM']
    chave = linha['Chave']
    veiculo = linha['Veículo']
    paradas = linha['Paradas']
    
    custo_fixo = tabela_final[
        (tabela_final['Chave'] == chave) &
        (tabela_final['KM Inicial'] <= km) &
        (tabela_final['KM Final'] >= km) &
        (tabela_final['Veículo'] == veiculo)
    ]
    
    if custo_fixo.empty:
        return None
    
    fixo = float(custo_fixo.iloc[0]['Custo Fixo'])
    
    tarifa = tarifas[tarifas['Região'] == chave]
    
    if tarifa.empty:
        return None
    
    v1 = float(tarifa.iloc[0]['Valor_0_60'])
    v2 = float(tarifa.iloc[0]['Valor_61_90'])
    v3 = float(tarifa.iloc[0]['Valor_91'])
    
    variavel = (
        min(paradas, 60) * v1 +
        max(min(paradas - 60, 30), 0) * v2 +
        max(paradas - 90, 0) * v3
    )
    
    return round(fixo + variavel, 2)

In [22]:
casos['Valor Total'] = casos.apply(calcular_linha, axis=1)

casos[['KM', 'Região', 'Veículo', 'Paradas', 'Valor Total']]

,KM,Região,Veículo,Paradas,Valor Total
0,248,Sudeste Capital,Van,36,532.6
1,111,Sul Interior,Risco,62,337.0
2,399,SP Interior 1,Vuc,111,711.8
3,300,SP Capital,Utilitario,74,517.2
4,150,Centro-Oeste Capital,Vuc,10,461.5
5,123,Centro-Oeste Interior,Utilitario,150,473.0
6,2,Sul Capital,Risco,122,378.0
7,245,Sul Interior,Utilitario,60,503.0
8,56,Sudeste Interior,Van,61,383.5
9,84,SP Interior 2,Utilitario,59,349.5


In [23]:
casos.to_excel("analise_final.xlsx", index=False)

In [24]:
tabela_final.to_excel("tabela_final.xlsx", index=False)